# Fact Tracing Benchmark Demo

This notebook demonstrates a how fact tracing benchmarks can be constructed for quanda using a fine-tuned NanoGPT and a subset of the TREx dataset.

### 1. Imports

In [ ]:
import os
import random
import tempfile

import datasets
import tiktoken
import torch

from quanda.benchmarks.downstream_eval import RecallAtK
from quanda.explainers.wrappers import Kronfluence
from quanda.utils.common import get_load_state_dict_func
from tests.conftest import LanguageModelingTaskNanoGPT
from tests.models import NanoGPT

## 2. Load Model

We use a GPT-2 model which was trained on 20 million sentences from T-REx and 10% of the Openwebtet dataset. Later, it was fine-tuned to predict brief answers to prompts like `The capital of France is: `

In [ ]:
# Load fine-tuned model from HuggingFace
model = NanoGPT.from_pretrained(
    "quanda-bench-test/gpt2-small-trex-openwebtext-ft",
)
model.eval()
print(
    f"Loaded NanoGPT model with {sum(p.numel() for p in model.parameters()):,} parameters"
)

## 3. Load Dataset

We use a subset of the T-REx dataset. Specifically, we select a subset with around 1500 prompts for which the above model was able to predict the answers correctly. (Therefore, we know the model has learned these facts from its pretraining.)

In [ ]:
# Load TREx subset from HuggingFace
dataset = datasets.load_dataset(
    "quanda-bench-test/trex-subset-benchmark", split="train"
)
print(f"Loaded TREx dataset with {len(dataset)} examples")
print(f"Sample entry: {dataset[0]}")

## 3. Prepare Dataset for Benchmark

In this small example, we select 20 prompts with up to 5 evidence sentences per prompt. In total this yields us 61 evidence sentences that we then rank via the specified TDA method (in our case Kronfluence). Note, that the model was trained on the entire set of evidence sentences (i.e. 20 million).

In [ ]:
# This function is copied from conftest.py


def _build_entailment_matrix(num_queries, num_evidence, evidence_map):
    """Build a binary entailment matrix from an evidence map."""
    entailment_labels = torch.zeros(
        (num_queries, num_evidence), dtype=torch.long
    )
    for j, query_idx in enumerate(evidence_map):
        entailment_labels[query_idx, j] = 1
    return entailment_labels


def load_fact_tracing_dataset_nanogpt(
    dataset_name="quanda-bench-test/trex-subset-benchmark",
    num_prompts=20,
    max_evidence_per_prompt=5,
    max_length=64,
    seed=1,
):
    # Load tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    pad_token_id = tokenizer.eot_token

    # Load dataset
    dataset = datasets.load_dataset(dataset_name, split="train")

    # Sample prompts
    if num_prompts < len(dataset):
        random.seed(seed)
        indices = random.sample(range(len(dataset)), num_prompts)
        sampled_dataset = dataset.select(indices)
    else:
        sampled_dataset = dataset

    input_ids = []
    attention_mask = []
    labels = []
    prompts = []
    answers = []

    for entry in sampled_dataset:
        prompt = entry["prompt"].strip()
        answer = entry["answer"][0].strip()
        full_text = prompt + " " + answer

        # Tokenize
        full_ids = tokenizer.encode_ordinary(full_text)[:max_length]
        prompt_ids = tokenizer.encode_ordinary(prompt)[:max_length]
        prompt_len = len(prompt_ids)

        # Pad
        padding_length = max_length - len(full_ids)
        padded_ids = full_ids + [pad_token_id] * padding_length
        padded_mask = [1] * len(full_ids) + [0] * padding_length

        # Create labels and mask out prompt and padding
        label_ids = padded_ids.copy()
        for i in range(min(prompt_len, max_length)):
            label_ids[i] = -100

        for i in range(len(full_ids), max_length):
            label_ids[i] = -100

        input_ids.append(padded_ids)
        attention_mask.append(padded_mask)
        labels.append(label_ids)
        prompts.append(prompt)
        answers.append(answer)

    prompt_dataset = datasets.Dataset.from_dict(
        {
            "prompt": prompts,
            "answer": answers,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }
    )

    # Collect evidence sentences
    evidence_sentences = []
    evidence_map = []

    for i, entry in enumerate(sampled_dataset):
        selected = entry["evidence_sentences"][:max_evidence_per_prompt]
        for sentence in selected:
            evidence_sentences.append(sentence)
            evidence_map.append(i)

    # Tokenize
    evidence_input_ids = []
    evidence_attention_mask = []
    evidence_labels = []
    for sentence in evidence_sentences:
        sentence_ids = tokenizer.encode_ordinary(sentence)[:max_length]
        padded_ids = sentence_ids + [pad_token_id] * (
            max_length - len(sentence_ids)
        )
        padded_mask = [1] * len(sentence_ids) + [0] * (
            max_length - len(sentence_ids)
        )

        # Create labels and mask out padding
        label_ids = padded_ids.copy()
        for i in range(len(sentence_ids), max_length):
            label_ids[i] = -100

        evidence_input_ids.append(padded_ids)
        evidence_attention_mask.append(padded_mask)
        evidence_labels.append(label_ids)

    evidence_dataset = datasets.Dataset.from_dict(
        {
            "sentence": evidence_sentences,
            "input_ids": evidence_input_ids,
            "attention_mask": evidence_attention_mask,
            "labels": evidence_labels,
        }
    )

    # Create entailment matrix
    entailment_labels = _build_entailment_matrix(
        len(prompt_dataset), len(evidence_dataset), evidence_map
    )

    return prompt_dataset, evidence_dataset, entailment_labels


# Prepare the dataset
prompt_dataset, evidence_dataset, entailment_labels = (
    load_fact_tracing_dataset_nanogpt()
)

print(f"Prompt dataset size: {len(prompt_dataset)}")
print(f"Evidence dataset size: {len(evidence_dataset)}")
print(f"Entailment labels shape: {entailment_labels.shape}")
print(f"Sample prompt: {prompt_dataset[0]['prompt']}")
print(f"Sample answer: {prompt_dataset[0]['answer']}")
print(f"Sample evidence: {evidence_dataset[0]['sentence']}")

## 4. Configure Benchmark

In this example we use the Recall@k metric, we could also use the MRR or Tail-Patch metric.

In [ ]:
# Setup benchmark
recall_benchmark = RecallAtK()
recall_benchmark.train_dataset = evidence_dataset
recall_benchmark.model = model
recall_benchmark.device = "cpu"
recall_benchmark.k = 3


with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as tmp_file:
    checkpoint_path = tmp_file.name
torch.save(model.state_dict(), checkpoint_path)

recall_benchmark.checkpoints = [checkpoint_path]
recall_benchmark.checkpoints_load_func = get_load_state_dict_func("cpu")
recall_benchmark.entailment_labels = entailment_labels
recall_benchmark.eval_dataset = prompt_dataset

# Setup explainer
task = LanguageModelingTaskNanoGPT()
expl_kwargs = {
    "task_module": task,
    "task": "causal_lm",
    "batch_size": 1,
    "cache_dir": "./cache",
}

print(f"Benchmark configured with k={recall_benchmark.k}")

## 5. Run Benchmark

In [ ]:
# Run evaluation
print("Running Recall@K benchmark...")
result = recall_benchmark.evaluate(
    explainer_cls=Kronfluence,
    expl_kwargs=expl_kwargs,
    batch_size=1,
)

score = result["score"]
print(f"Recall@{recall_benchmark.k} Score: {score:.4f}")

os.unlink(checkpoint_path)

## 6. Interpretation

The Recall@K score measures the proportion of queries for which at least one entailing training example appears in the top-K retrieved proponents according to the selected influence attribution method.

- **Score = 1.0**: Every query has at least one entailing example in its top-K proponents
- **Score = 0.0**: No entailing examples found in top-K for any query

This benchmark helps evaluate how well TDA methods can identify training examples that logically support given facts.